# Lab Day 1 — Neural Network experiments

Notebook nộp bài cho MSSV 2A202602813. Pipeline dùng đúng split cố định trong `data/split_metadata.csv`, chọn cấu hình chỉ bằng validation, rồi mới chấm eval. Chạy từ đầu đến cuối trên Colab/Kaggle có GPU để chạy thí nghiệm mixed precision.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, shutil, csv, math, time
import numpy as np
import pandas as pd
import torch

os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next((p for p in candidates if (p / "scripts" / "evaluate.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run the notebook from within the cloned repository.")
SUBMISSION_DIR = REPO_ROOT / "submission_2A202602813"
OUT_DIR = SUBMISSION_DIR
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
package_code = SUBMISSION_DIR / "code"
CODE_DIR = package_code if (package_code / "report.py").exists() else REPO_ROOT / "code"
sys.path.insert(0, str(CODE_DIR))
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

from data import N_NUMERIC, prepare_data, load_eval_labels
from model import MLP, EXPECTED_PARAMS, count_params, activation_stats
from train import DEFAULT_CFG, evaluate, run_experiment, set_seed, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, to_row, write_xlsx
from report import build_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not (PROCESSED_DIR / "train.npz").exists() or not (PROCESSED_DIR / "eval.npz").exists():
    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    split = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT,
                          env=env, text=True, capture_output=True, check=True)
    print(split.stdout)
print("Repository:", REPO_ROOT)
print("Output:", OUT_DIR)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("PyTorch:", torch.__version__)

Repository: C:\Users\ADMIN\Documents\PRJ\day01
Output: C:\Users\ADMIN\Documents\PRJ\day01\submission_2A202602813
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
PyTorch: 2.7.1+cu118


## Part 0 — Fixed data split and train-only standardization

Tách validation từ train theo stratification và seed 42. Nhãn eval không được nạp vào pipeline huấn luyện.

In [3]:
data = prepare_data(device=device, val_fraction=0.2, seed=42,
                    processed_dir=PROCESSED_DIR)
assert "y_eval" not in data  # Chưa mở nhãn eval trước khi khóa cấu hình.
assert tuple(data["X_tr"].shape) == (371847, 54)
assert tuple(data["X_val"].shape) == (92962, 54)
assert tuple(data["X_eval"].shape) == (116203, 54)
majority_train_acc = data["majority_train_acc"]

train/val/eval: (371847, 54) / (92962, 54) / (116203, 54)
train-only normalization: max |mean|=0.00000; max |std-1|=0.00075
majority-class accuracy on train: 0.4876


## Part 1 — Model và kiểm tra ban đầu

M-base có shape 54 → 256 → 128 → 7, trả logits thô. Kiểm tra loss bước 0, gradient và khả năng overfit 20 mẫu trước khi chạy dài.

In [3]:
set_seed(42)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47879
logits = model(data["X_val"][:8])
assert tuple(logits.shape) == (8, 7)
step0 = evaluate(model, data["X_val"], data["y_val"], "ce")
initial_activation_std = activation_stats(model, data["X_val"][:4096])
model.zero_grad(set_to_none=True)
loss0 = torch.nn.functional.cross_entropy(model(data["X_tr"][:64]), data["y_tr"][:64])
loss0.backward()
gradient_pass = all(p.grad is not None and torch.isfinite(p.grad).all() and
                    torch.any(p.grad != 0) for p in model.parameters())
print("M-base parameters:", count_params(model), "logits:", tuple(logits.shape))
print(f"Initial validation CE loss: {step0['loss']:.4f}; ln(7)={math.log(7):.4f}")
print("Initial activation std after hidden ReLU:", initial_activation_std)
print("All parameters have finite, nonzero gradients:", gradient_pass)
assert gradient_pass

tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
tiny_optimizer = torch.optim.Adam(tiny.parameters(), lr=1e-2)
overfit_acc, overfit_loss, overfit_steps = 0.0, float("inf"), 0
for step in range(2500):
    tiny.train()
    tiny_optimizer.zero_grad(set_to_none=True)
    tiny_loss = torch.nn.functional.cross_entropy(tiny(data["X_tr"][:20]), data["y_tr"][:20])
    tiny_loss.backward()
    tiny_optimizer.step()
    if step % 10 == 0 or step == 2499:
        tiny_metrics = evaluate(tiny, data["X_tr"][:20], data["y_tr"][:20], "ce")
        overfit_acc, overfit_loss = tiny_metrics["acc"], tiny_metrics["loss"]
        overfit_steps = step + 1
        if overfit_acc >= 0.99:
            break
print(f"20-row overfit: acc={overfit_acc:.1%}, loss={overfit_loss:.5f}, steps={overfit_steps}")
assert overfit_acc >= 0.99
sanity = {
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "torch_version": torch.__version__,
    "majority_train_acc": majority_train_acc,
    "gradient_pass": bool(gradient_pass),
    "overfit_acc": float(overfit_acc),
    "overfit_loss": float(overfit_loss),
    "overfit_steps": int(overfit_steps),
}
del model, tiny, tiny_optimizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

M-base parameters: 47879 logits: (8, 7)
Initial validation CE loss: 2.3776; ln(7)=1.9459
Initial activation std after hidden ReLU: [0.4139760434627533, 0.3787032961845398]
All parameters have finite, nonzero gradients: True


20-row overfit: acc=100.0%, loss=0.01071, steps=11


## Part 2–3 — Baseline và controlled experiments

Mỗi cấu hình đổi một yếu tố so với baseline, ngoại trừ cặp clipping dùng cùng lr cao 1.0 để kiểm tra ổn định. Checkpoint mỗi run là epoch có validation loss thấp nhất.

In [4]:
base = {
    "loss": "ce", "optimizer": "sgd_momentum", "lr": 0.05, "weight_decay": 0.0,
    "momentum": 0.9, "batch": 512, "epochs": 20, "hidden": (256, 128),
    "dropout": 0.0, "init": "he", "clip_norm": None, "precision": "fp32",
    "eligible_for_best": True,
}
def experiment(exp_id, group, hypothesis, **changes):
    return {**base, "exp_id": exp_id, "group": group,
            "description": hypothesis, "hypothesis": hypothesis, **changes}

EXPERIMENTS = [
    experiment("baseline_s42", "baseline", "He + CE + SGD momentum should beat majority baseline.", seed=42),
    experiment("baseline_s123", "baseline", "A second seed estimates baseline variation.", seed=123),
    experiment("loss_mse", "loss", "CE should outperform MSE for seven-class classification.", loss="mse", seed=42),
    experiment("adam_1e3", "optimizer", "Adam at lr 1e-3 may converge faster than SGD momentum.",
               optimizer="adam", lr=1e-3, seed=42),
    experiment("adam_3e4", "optimizer", "A smaller Adam lr may trade speed for stability.",
               optimizer="adam", lr=3e-4, seed=42),
    experiment("adam_3e3", "optimizer", "Adam at 3e-3 may converge faster but could become less stable.",
               optimizer="adam", lr=3e-3, seed=42),
    experiment("sgd_1e3", "optimizer", "At matched lr=1e-3, compare SGD momentum directly with Adam.",
               lr=1e-3, seed=42),
    experiment("sgd_1e2", "optimizer", "A lower SGD momentum lr may trade stability for slower convergence.",
               lr=1e-2, seed=42),
    experiment("sgd_1e1", "optimizer", "A higher SGD momentum lr may speed learning but increase instability.",
               lr=1e-1, seed=42),
    experiment("width_512_256", "hparam", "A wider hidden network may improve validation F1.",
               hidden=(512, 256), seed=42),
    experiment("dropout_03", "dropout", "Dropout 0.3 may reduce overfitting.", dropout=0.3, seed=42),
    experiment("baseline_clip1", "clipping", "At the baseline lr, clip norm 1.0 may be inactive; logs will measure the activation rate.", clip_norm=1.0, eligible_for_best=False, seed=42),
    experiment("highlr_no_clip", "clipping", "At lr 1.0, no clipping may destabilize training.",
               lr=1.0, eligible_for_best=False, seed=42),
    experiment("highlr_clip1", "clipping", "Clipping at 1.0 may stabilize the same high-lr run.",
               lr=1.0, clip_norm=1.0, eligible_for_best=False, seed=42),
    experiment("init_xavier", "init", "Compare Xavier with He initialization for ReLU.",
               init="xavier", seed=42),
]
if torch.cuda.is_available():
    EXPERIMENTS.append(experiment(
        "amp_fp16", "amp", "FP16 may reduce memory or time with similar validation quality.",
        precision="fp16", seed=42
    ))
else:
    print("FP16 experiment skipped: CUDA is not available on this runtime.")

results_dir = OUT_DIR / "results"
figures_dir = OUT_DIR / "figures"
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
results = []
for index, cfg in enumerate(EXPERIMENTS, start=1):
    print(f"\n[{index}/{len(EXPERIMENTS)}] {cfg['exp_id']}")
    print("Dự đoán trước:", cfg["hypothesis"])
    result = run_experiment(cfg, data)
    results.append(result)
    save_result(result, results_dir)
    plot_run(result, figures_dir / f"{cfg['exp_id']}.png")
    score = result["summary"].get("val_macro_f1")
    reference = next((r for r in results if r["cfg"]["exp_id"] == "baseline_s42"), None)
    if cfg["exp_id"] == "baseline_s42" or reference is None:
        observation = f"val macro-F1={score:.4f}; ghi nhận mốc baseline."
    else:
        base_score = reference["summary"].get("val_macro_f1")
        delta = score - base_score if score is not None and base_score is not None else float("nan")
        direction = "cao hơn" if delta > 0 else "thấp hơn" if delta < 0 else "bằng"
        observation = (f"val macro-F1={score:.4f}, {direction} baseline_s42 {abs(delta):.4f}; "
                       f"checkpoint val-loss tốt nhất ở epoch {result['summary']['best_epoch']}; "
                       f"diverged={result['summary']['diverged']}.")
    print("Nhận xét sau chạy:", observation)
    print("Chi tiết:", result["summary"])

by_id = {result["cfg"]["exp_id"]: result for result in results}
plot_compare([by_id["baseline_s42"], by_id["baseline_s123"]], "val_macro_f1",
             figures_dir / "compare_baseline.png", "Baseline seed variation")
plot_compare([by_id[name] for name in (
    "baseline_s42", "sgd_1e3", "sgd_1e2", "sgd_1e1",
    "adam_3e4", "adam_1e3", "adam_3e3",
)], "val_macro_f1", figures_dir / "compare_optimizer.png",
             "Validation macro-F1 across SGD momentum and Adam learning-rate grids")
plot_compare([by_id["baseline_s42"], by_id["baseline_clip1"],
              by_id["highlr_no_clip"], by_id["highlr_clip1"]], "val_macro_f1",
             figures_dir / "compare_clipping.png", "Validation macro-F1: clipping at lr=0.05 and lr=1.0")
summary_rows = [{
    "exp_id": r["cfg"]["exp_id"], "group": r["cfg"]["group"],
    "optimizer": r["cfg"]["optimizer"], "lr": r["cfg"]["lr"],
    "best_epoch_by_val_loss": r["summary"]["best_epoch"],
    "best_val_loss": r["summary"]["best_val_loss"],
    "val_acc": r["summary"]["val_acc"],
    "val_macro_f1": r["summary"]["val_macro_f1"],
    "diverged": r["summary"]["diverged"],
} for r in results]
display(pd.DataFrame(summary_rows))


[1/16] baseline_s42
Dự đoán trước: He + CE + SGD momentum should beat majority baseline.


C:\Users\ADMIN\Documents\PRJ\day01\submission_2A202602813\code\train.py:127: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)


baseline_s42         epoch=01 train_loss=0.4987 val_loss=0.5021 val_macro_f1=0.5915


baseline_s42         epoch=20 train_loss=0.2313 val_loss=0.2491 val_macro_f1=0.8453


Nhận xét sau chạy: val macro-F1=0.8342; ghi nhận mốc baseline.
Chi tiết: {'step0_loss': 2.3775719869378147, 'activation_std_step0': [0.4139760434627533, 0.3787032961845398], 'best_val_loss': 0.24840128978192633, 'best_epoch': 18, 'final_train_loss': 0.23131974115371703, 'final_val_loss': 0.2490982591306965, 'val_acc': 0.9010240435600281, 'val_macro_f1': 0.8341818718902233, 'time_per_epoch_s': 1.7951530400008777, 'total_time_s': 35.916597999996156, 'peak_mem_MB': 159.76025390625, 'diverged': False, 'param_count': 47879}

[2/16] baseline_s123
Dự đoán trước: A second seed estimates baseline variation.


...

Nhận xét sau chạy: val macro-F1=0.8489, cao hơn baseline_s42 0.0147; checkpoint val-loss tốt nhất ở epoch 20; diverged=False.
Chi tiết: {'step0_loss': 2.0429060532846113, 'activation_std_step0': [0.17277909815311432, 0.12905341386795044], 'best_val_loss': 0.2421080537433866, 'best_epoch': 20, 'final_train_loss': 0.22690716953277587, 'final_val_loss': 0.2421080537433866, 'val_acc': 0.90208899974823, 'val_macro_f1': 0.8488955415545257, 'time_per_epoch_s': 1.5200824100000319, 'total_time_s': 30.41246000000683, 'peak_mem_MB': 159.76025390625, 'diverged': False, 'param_count': 47879}

[16/16] amp_fp16
Dự đoán trước: FP16 may reduce memory or time with similar validation quality.


amp_fp16             epoch=01 train_loss=0.4985 val_loss=0.5018 val_macro_f1=0.5923


amp_fp16             epoch=20 train_loss=0.2327 val_loss=0.2531 val_macro_f1=0.8427


Nhận xét sau chạy: val macro-F1=0.8375, cao hơn baseline_s42 0.0034; checkpoint val-loss tốt nhất ở epoch 18; diverged=False.
Chi tiết: {'step0_loss': 2.3775719869378147, 'activation_std_step0': [0.4139760434627533, 0.3787032961845398], 'best_val_loss': 0.2507148684391951, 'best_epoch': 18, 'final_train_loss': 0.2326549698162079, 'final_val_loss': 0.2531084662596224, 'val_acc': 0.9001204371452332, 'val_macro_f1': 0.8375340758604963, 'time_per_epoch_s': 3.1637960200001545, 'total_time_s': 63.29120609999518, 'peak_mem_MB': 159.75927734375, 'diverged': False, 'param_count': 47879}


,exp_id,group,optimizer,lr,best_epoch_by_val_loss,best_val_loss,val_acc,val_macro_f1,diverged
0,baseline_s42,baseline,sgd_momentum,0.0500,18,0.248401,0.901024,0.834182,False
1,baseline_s123,baseline,sgd_momentum,0.0500,18,0.256892,0.897119,0.835984,False
2,loss_mse,loss,sgd_momentum,0.0500,20,0.031792,0.852531,0.702888,False
3,adam_1e3,optimizer,adam,0.0010,20,0.246879,0.899163,0.852070,False
4,adam_3e4,optimizer,adam,0.0003,20,0.310691,0.874045,0.791705,False
5,adam_3e3,optimizer,adam,0.0030,20,0.208495,0.916643,0.875707,False
6,sgd_1e3,optimizer,sgd_momentum,0.0010,20,0.520803,0.780577,0.593589,False
7,sgd_1e2,optimizer,sgd_momentum,0.0100,20,0.325005,0.866074,0.744862,False
8,sgd_1e1,optimizer,sgd_momentum,0.1000,18,0.231674,0.907166,0.845345,False
9,width_512_256,hparam,sgd_momentum,0.0500,20,0.219682,0.912889,0.863706,False


## Part 4a — Chốt cấu hình bằng validation

Chọn run có macro-F1 cao nhất tại checkpoint val-loss thấp nhất, loại trừ hai run lr=1.0 vốn chỉ dùng để kiểm tra clipping. Sau bước này mới nạp nhãn eval để chấm.

In [5]:
eligible = [r for r in results
            if r["cfg"].get("eligible_for_best", True)
            and not r["summary"]["diverged"]
            and r["summary"].get("val_macro_f1") is not None]
selected_result = max(eligible, key=lambda r: r["summary"]["val_macro_f1"])
selected_id = selected_result["cfg"]["exp_id"]
print("Locked final model by validation only:", selected_id)
print("Validation macro-F1 at best val-loss epoch:", selected_result["summary"]["val_macro_f1"])
print("Checkpoint epoch:", selected_result["summary"]["best_epoch"])

Locked final model by validation only: adam_3e3
Validation macro-F1 at best val-loss epoch: 0.8757065101471024
Checkpoint epoch: 20


## Part 4b — Eval, bảng, lỗi theo lớp và report

Sau khi đã khóa cấu hình, chấm baseline_s42 để so sánh và cấu hình cuối cùng bằng đúng scripts/evaluate.py. Chỉ predictions của cấu hình cuối được đặt ở thư mục gốc submission.

In [6]:
eval_dir = OUT_DIR / "results" / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)
subprocess_env = os.environ.copy()
subprocess_env["PYTHONUTF8"] = "1"
subprocess_env["PYTHONIOENCODING"] = "utf-8"

def score_predictions(pred_path, out_path):
    proc = subprocess.run(
        [sys.executable, "scripts/evaluate.py", "--pred", str(pred_path), "--out", str(out_path)],
        cwd=REPO_ROOT, env=subprocess_env, text=True, capture_output=True, check=True
    )
    print(proc.stdout)
    return json.loads(Path(out_path).read_text(encoding="utf-8"))

baseline_result = by_id["baseline_s42"]
baseline_pred_path = eval_dir / "baseline_predictions_eval.csv"
final_eval({**DEFAULT_CFG, **baseline_result["cfg"]}, baseline_result, data, baseline_pred_path)
baseline_eval_path = eval_dir / "baseline_eval_result.json"
baseline_eval = score_predictions(baseline_pred_path, baseline_eval_path)

final_pred_path = OUT_DIR / "predictions_eval.csv"
final_eval({**DEFAULT_CFG, **selected_result["cfg"]}, selected_result, data, final_pred_path)
final_eval_path = OUT_DIR / "eval_result.json"
final_scores = score_predictions(final_pred_path, final_eval_path)

# Eval labels are first loaded here, after validation has locked the selected model.
y_eval = load_eval_labels(PROCESSED_DIR)
X_eval_cpu = data["X_eval"].detach().cpu().numpy()
feature_names = [str(name) for name in data["feature_names"]]
cm = np.asarray(final_scores["confusion_matrix"], dtype=int)
hardest_class = int(np.argmin([row["f1"] for row in final_scores["per_class"]]))
partner_counts = cm[hardest_class].copy()
partner_counts[hardest_class] = 0
confused_class = int(partner_counts.argmax())
mean_hard = X_eval_cpu[y_eval == hardest_class].mean(axis=0)
mean_partner = X_eval_cpu[y_eval == confused_class].mean(axis=0)
mean_delta = np.abs(mean_hard - mean_partner)
close_indices = np.argsort(mean_delta[:N_NUMERIC])[:3]
error_context = {"closest_mean_features": [
    {"name": feature_names[i], "abs_delta": float(mean_delta[i])} for i in close_indices
]}
print("Most difficult class:", hardest_class, "most common wrong prediction:", confused_class)
print("Closest continuous class-mean features:", error_context["closest_mean_features"])

baseline_values = np.asarray([by_id[name]["summary"]["val_macro_f1"]
                              for name in ("baseline_s42", "baseline_s123")], dtype=float)
two_sigma = 2 * baseline_values.std(ddof=1)
rows = []
for result in results:
    exp_id = result["cfg"]["exp_id"]
    eval_scores = None
    if exp_id == "baseline_s42":
        eval_scores = baseline_eval
    if exp_id == selected_id:
        eval_scores = final_scores
    val_f1 = result["summary"].get("val_macro_f1")
    delta = None if val_f1 is None else float(val_f1) - float(by_id["baseline_s42"]["summary"]["val_macro_f1"])
    notes = result["cfg"].get("hypothesis", "")
    if delta is not None:
        notes += f" Observed val macro-F1 delta vs baseline_s42: {delta:+.4f}; 2-sigma threshold: {two_sigma:.4f}."
    rows.append(to_row(result, eval_scores=eval_scores, notes=notes))

summary_notes = {
    "baseline": f"Two seeds; val macro-F1 mean={baseline_values.mean():.4f}, sample SD={baseline_values.std(ddof=1):.4f}.",
    "loss": "MSE on softmax probabilities vs CE; compare validation macro-F1, not loss scales.",
    "optimizer": "SGD momentum grid 0.001/0.01/0.05/0.1; Adam grid 0.0003/0.001/0.003; matched-lr comparison at 0.001.",
    "hparam": "Wider network changes layer width and parameter count together.",
    "dropout": "q=0.3; compare train/validation loss gap and validation macro-F1.",
    "clipping": "Clip norm 1.0 measured at baseline lr=0.05 and paired with/without clipping at lr=1.0.",
    "amp": "FP16 on this CUDA GPU; speed and memory are hardware-specific.",
    "init": "He vs Xavier; activation standard deviations logged at step 0.",
}
write_xlsx(rows, REPO_ROOT / "templates" / "experiment_table_template.xlsx",
           OUT_DIR / "experiments.xlsx", seed_ids=["baseline_s42", "baseline_s123"],
           summary_notes=summary_notes)

report_text = build_report(
    results, sanity, baseline_eval, final_scores, selected_id,
    OUT_DIR / "REPORT.md", error_context=error_context
)
print("Report characters:", len(report_text))

n_eval = 116203
accuracy = 0.9000
macro_f1 = 0.8355   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9236  0.8743  0.8983
  1    56661     0.8976  0.9417  0.9191
  2     7151     0.8276  0.9168  0.8699
  3      549     0.8184  0.7304  0.7719
  4     1899     0.8358  0.6672  0.7420
  5     3473     0.8057  0.6720  0.7328
  6     4102     0.9474  0.8832  0.9142

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  37042   5091     7    0    38    12   178
1   2622  53360   288    0   200   168    23
2      1    187  6556   57     4   346     0
3      0      0   126  401     0    22     0
4     25    533    59    0  1267    15     0
5      8    206   886   32     7  2334     0
6    408     71     0    0     0     0  3623

đã ghi C:\Users\ADMIN\Documents\PRJ\day01\submission_2A202602813\results\evaluation\baseline_eval_result.json



n_eval = 116203
accuracy = 0.9168
macro_f1 = 0.8810   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9008  0.9263  0.9134
  1    56661     0.9359  0.9191  0.9274
  2     7151     0.9213  0.9150  0.9181
  3      549     0.8373  0.8342  0.8358
  4     1899     0.7910  0.7973  0.7941
  5     3473     0.8526  0.8477  0.8501
  6     4102     0.9424  0.9137  0.9278

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  39247   2850     1    0    57     8   205
1   3961  52077   175    1   308   116    23
2      1    172  6543   66    31   338     0
3      0      0    51  458     0    40     0
4     51    305    21    0  1514     7     1
5      7    185   311   22     4  2944     0
6    301     53     0    0     0     0  3748

đã ghi C:\Users\ADMIN\Documents\PRJ\day01\submission_2A202602813\eval_result.json

Most difficult class: 4 most common wrong prediction: 1
Closest continuous class-mean features: [{'name': 'Vertical_Dis

## Tự kiểm tra đầu ra

Notebook kiểm tra CSV, kết quả evaluator, bảng từ template, biểu đồ và các file code không còn phần khung chưa hoàn thiện.

In [5]:
with final_pred_path.open("r", newline="", encoding="utf-8") as handle:
    prediction_rows = list(csv.DictReader(handle))
assert len(prediction_rows) == 116203
assert list(prediction_rows[0].keys()) == ["row_id", "pred"]
row_ids = np.asarray([int(row["row_id"]) for row in prediction_rows])
pred_labels = np.asarray([int(row["pred"]) for row in prediction_rows])
assert len(np.unique(row_ids)) == 116203
assert set(row_ids.tolist()) == set(data["eval_row_id"].tolist())
assert pred_labels.min() >= 0 and pred_labels.max() <= 6
assert final_scores["n_eval"] == 116203
assert len(list(figures_dir.glob("*.png"))) == len(results) + 3
assert all((figures_dir / f"{r['cfg']['exp_id']}.png").exists() for r in results)

from openpyxl import load_workbook
wb = load_workbook(OUT_DIR / "experiments.xlsx", data_only=False)
assert wb.sheetnames == ["Legend", "Experiments", "Seeds", "Summary"]
ws = wb["Experiments"]
assert [ws.cell(1, c).value for c in range(1, 30)] == [
    "exp_id", "group", "description", "loss", "optimizer", "lr", "weight_decay", "batch",
    "epochs", "hidden", "dropout", "clip_norm", "precision", "init", "seed", "step0_loss",
    "best_val_loss", "best_epoch", "final_train_loss", "final_val_loss", "val_acc",
    "val_macro_f1", "time_per_epoch_s", "peak_mem_MB", "diverged", "eval_acc",
    "eval_macro_f1", "figure_file", "notes"
]
assert [ws.cell(i, 1).value for i in range(2, 2 + len(results))] == [r["cfg"]["exp_id"] for r in results]
assert ws["AB2"].value == "figures/baseline_s42.png"
source_files = [*CODE_DIR.glob("*.py"), CODE_DIR / "lab.ipynb"]
placeholder_marker = "NotImplemented" + "Error"
incomplete_files = [p.name for p in source_files
                    if placeholder_marker in p.read_text(encoding="utf-8", errors="replace")]
assert not incomplete_files, f"unfinished code scaffold: {incomplete_files}"
wb_values = load_workbook(OUT_DIR / "experiments.xlsx", data_only=True)
formula_errors = [(sheet.title, cell.coordinate, cell.value)
                  for sheet in wb_values.worksheets for row in sheet.iter_rows()
                  for cell in row if isinstance(cell.value, str) and cell.value.startswith("#")]
assert not formula_errors, f"workbook formula errors: {formula_errors}"
assert final_eval_path.exists() and (OUT_DIR / "REPORT.md").exists()
shutil.rmtree(CODE_DIR / "__pycache__", ignore_errors=True)
print("PASS: predictions, official eval, workbook sheets/columns, figures, and report.")
print("Submission files:")
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(OUT_DIR))

PASS: predictions, official eval, workbook sheets/columns, figures, and report.
Submission files:
- code\data.py
- code\lab.ipynb
- code\model.py
- code\optimizer.py
- code\plots.py
- code\report.py
- code\results_table.py
- code\train.py
- eval_result.json
- experiments.xlsx
- figures\adam_1e3.png
- figures\adam_3e3.png
- figures\adam_3e4.png
- figures\amp_fp16.png
- figures\baseline_clip1.png
- figures\baseline_s123.png
- figures\baseline_s42.png
- figures\compare_baseline.png
- figures\compare_clipping.png
- figures\compare_optimizer.png
- figures\dropout_03.png
- figures\highlr_clip1.png
- figures\highlr_no_clip.png
- figures\init_xavier.png
- figures\loss_mse.png
- figures\sgd_1e1.png
- figures\sgd_1e2.png
- figures\sgd_1e3.png
- figures\width_512_256.png
- predictions_eval.csv
- REPORT.md
- results\adam_1e3.json
- results\adam_3e3.json
- results\adam_3e4.json
- results\amp_fp16.json
- results\baseline_clip1.json
- results\baseline_s123.json
- results\baseline_s42.json
- results\d